# Basketbal Vlaanderen (VBL) — Foutenanalyse & Technische Fouten per Speler

Dit notebook analyseert wedstrijdverslagen van **Basketbal Vlaanderen (VBL / Wisseq)** voor een gekozen club en ploeg.
Het automatiseert de volledige flow:

1. **Club inladen**: Leest de clubgegevens op basis van de club-GUID (bv. `BVBL1037` voor BBC Haantjes Oudenaarde).
2. **Ploeg selecteren**: Lijst alle ploegen van de club op en laat een specifieke ploeg selecteren (bv. `BVBL1037HSE  1` voor HSE A).
3. **Voorbije wedstrijden (Uitslagen)**: Haalt alle gespeelde wedstrijden op van het huidige seizoen.
4. **Wedstrijdverslagen (DWF)**: Doorloopt elke wedstrijd-GUID.
5. **Uitgebreid verslag (Matchdetailpdf)**: Vraagt via de VBL WebService de digitale wedstrijdbladen op:
   - Deelnemers / Spelerslijst (`DwfDeelByWedGuid`)
   - Voortgang / Gebeurtenissen (`DwfVgngByWedGuid`)
6. **Fouten per speler per type**: Brengt naast de naam van elke speler alle gemaakte fouten in kaart, uitgesplitst per type:
   - **Persoonlijke fouten** (`P`, `P1`, `P2`, `P3`)
   - **Technische fouten** (`T`, `T1`, `FL`, `FL2`, `C`, `C1`, `B`, `B1`)
   - **Onsportieve fouten** (`U`, `U1`, `U2`)
   - **Diskwalificerende fouten** (`D`, `D1`, `D2`, `DI`, `DI2`, `GD`)

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt

# VBL API Base URL & Headers (publiek toegankelijk, conform Wisseq / VBL DWF)
VBL_BASE_URL = "https://vblcb.wisseq.eu/VBLCB_WebService/data"

API_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "authorization": "na",
    "ssoBearer": "ssoWQAndre",
    "Content-Type": "application/json; charset=utf-8",
}

print("Setup voltooid. Klaar om VBL data op te halen!")

## Stap 1 & 2: Club inladen en ploeg selecteren

We halen alle ploegen van de club op via het `OrgMatchesByGuid` endpoint. We nemen standaard **BBC Haantjes Oudenaarde (`BVBL1037`)** als voorbeeld.

In [ ]:
CLUB_GUID = "BVBL1037"  # BBC Haantjes Certifisc Oudenaarde

def get_club_teams(club_guid: str) -> pd.DataFrame:
    """Haalt alle unieke ploegen van een club op uit het wedstrijdprogramma."""
    url = f"{VBL_BASE_URL}/OrgMatchesByGuid?issguid={club_guid}"
    resp = requests.get(url, headers=API_HEADERS, timeout=15)
    resp.raise_for_status()
    matches = resp.json()
    
    teams = {}
    for m in matches:
        if club_guid in m.get("tTGUID", ""):
            teams[m.get("tTGUID")] = m.get("tTNaam")
        if club_guid in m.get("tUGUID", ""):
            teams[m.get("tUGUID")] = m.get("tUNaam")
            
    df_teams = pd.DataFrame([
        {"team_guid": guid, "team_naam": naam}
        for guid, naam in sorted(teams.items())
    ])
    return df_teams

df_teams = get_club_teams(CLUB_GUID)
print(f"Gevonden ploegen voor {CLUB_GUID} ({len(df_teams)} ploegen):")
display(df_teams)

### Selecteer de gewenste ploeg
Hieronder kiezen we de ploeg waarvan we de wedstrijden en fouten willen analyseren. Standaard staat dit op **HSE A** (`BVBL1037HSE  1`).

In [ ]:
# Kies hier de gewenste team_guid uit de tabel hierboven:
SELECTED_TEAM_GUID = "BVBL1037HSE  1"  # bv. HSE A

selected_team_name = df_teams.loc[df_teams["team_guid"] == SELECTED_TEAM_GUID, "team_naam"].values[0]
print(f"Geselecteerde ploeg: {selected_team_name} ({SELECTED_TEAM_GUID})")

## Stap 3: Voorbije wedstrijden (Uitslagen) ophalen

We roepen het `TeamManage` endpoint aan met commando `GETWEDS` om de volledige wedstrijdkalender van deze ploeg op te halen, en filteren op de reeds gespeelde wedstrijden (uitslagen).

In [ ]:
def get_team_played_matches(team_guid: str) -> pd.DataFrame:
    """Haalt alle gespeelde wedstrijden (tab uitslagen) op voor een ploeg."""
    payload = {
        "AuthHeader": "na",
        "WQVer": "nov2",
        "Command": "GETWEDS",
        "TeamGUID": team_guid,
        "CRUD": "R",
    }
    resp = requests.put(f"{VBL_BASE_URL}/TeamManage", headers=API_HEADERS, json=payload, timeout=15)
    resp.raise_for_status()
    all_matches = resp.json().get("wedList", [])
    
    # Filter op gespeelde wedstrijden (waar een uitslag voor gekend is)
    played = [m for m in all_matches if m.get("uitslag") and str(m.get("uitslag")).strip()]
    df = pd.DataFrame(played)
    
    cols = ["guid", "datumString", "beginTijd", "tTNaam", "tUNaam", "uitslag", "pouleNaam"]
    return df[[c for c in cols if c in df.columns]].rename(columns={
        "guid": "wedguid",
        "datumString": "datum",
        "beginTijd": "uur",
        "tTNaam": "thuisploeg",
        "tUNaam": "uitploeg",
        "pouleNaam": "reeks",
    })

df_played = get_team_played_matches(SELECTED_TEAM_GUID)
print(f"Aantal gespeelde wedstrijden: {len(df_played)}")
display(df_played)

## Stap 4 & 5: Wedstrijdverslagen ophalen via DWF (Matchdetailpdf)

Wanneer je op VBL Web op een wedstrijd klikt en het **uitgebreid verslag** (`Matchdetailpdf`) opent, haalt de webapp twee datastromen op:
1. `DwfDeelByWedGuid`: Deelnemerslijst (spelers, rugnummers, coaches) van beide teams (`TtDeel` en `TuDeel`).
2. `DwfVgngByWedGuid`: Chronologische wedstrijdgebeurtenissen (`GebNis`), waarbij `GebType == 30` alle fouten voorstelt.

Onderstaande functie combineert deze twee stromen en normaliseert alle foutcodes (zoals `P`, `P1`, `P2`, `P3`, `(T1)`, `(FL2)`, `C1`, `(B1)`, `DI`, `DI2`).

In [ ]:
# Betekenis van alle mogelijke foutcodes in Basketbal Vlaanderen / FIBA DWF
FOUL_EXPLANATIONS = {
    "P": "Persoonlijke fout zonder vrijworpen (balinworp)",
    "P1": "Persoonlijke fout met 1 vrijworp (and-one)",
    "P2": "Persoonlijke fout met 2 vrijworpen (op 2-punter of teamfouten)",
    "P3": "Persoonlijke fout met 3 vrijworpen (op driepunter)",
    "T": "Technische fout speler (1 vrijworp)",
    "T1": "Technische fout speler (1 vrijworp)",
    "FL": "Flopping technische fout (theatrale val, 1 vrijworp)",
    "FL2": "Flopping technische fout (1 vrijworp)",
    "C": "Technische fout coach direct (voor eigen gedrag)",
    "C1": "Technische fout coach direct (voor eigen gedrag)",
    "B": "Technische fout bank / assistent / reservespelers",
    "B1": "Technische fout bank / assistent / reservespelers",
    "U": "Onsportieve fout (2 vrijworpen + balbezit)",
    "U1": "Onsportieve fout met 1 vrijworp + balbezit",
    "U2": "Onsportieve fout met 2 vrijworpen + balbezit",
    "D": "Diskwalificerende fout (onmiddellijke uitsluiting)",
    "D1": "Diskwalificerende fout met 1 vrijworp",
    "D2": "Diskwalificerende fout met 2 vrijworpen",
    "DI": "Diskwalificerende fout (onmiddellijke uitsluiting)",
    "DI2": "Diskwalificerende fout met 2 vrijworpen",
    "GD": "Game Disqualification (uitsluiting na 5 fouten of 2x T/U)",
}

def parse_match_fouls(wedguid: str) -> list:
    """Haalt deelnemers en fouten op voor één wedstrijd-GUID."""
    payload = {
        "AuthHeader": "na",
        "WQVer": "nov2",
        "WedGUID": wedguid,
        "CRUD": "R",
    }
    
    try:
        r_deel = requests.put(f"{VBL_BASE_URL}/DwfDeelByWedGuid", headers=API_HEADERS, json=payload, timeout=10)
        r_vgng = requests.put(f"{VBL_BASE_URL}/DwfVgngByWedGuid", headers=API_HEADERS, json=payload, timeout=10)
    except Exception as e:
        print(f"Fout bij ophalen van {wedguid}: {e}")
        return []
        
    if r_deel.status_code != 200 or r_vgng.status_code != 200:
        return []
        
    try:
        data_deel = r_deel.json() if r_deel.text else {}
    except Exception:
        data_deel = {}

    try:
        data_vgng = r_vgng.json() if r_vgng.text else {}
    except Exception:
        data_vgng = {}

    if not isinstance(data_deel, dict):
        data_deel = {}
    if not isinstance(data_vgng, dict):
        data_vgng = {}
        
    # Bouw opzoekwoordenboek van spelers op basis van RelGUID
    players = {}
    for p in (data_deel.get("TtDeel") or []):
        if isinstance(p, dict):
            players[p.get("RelGUID")] = {
                "naam": p.get("Naam"),
                "rugnr": p.get("RugNr"),
                "team_kant": "Thuis",
                "functie": p.get("Functie"),
            }
    for p in (data_deel.get("TuDeel") or []):
        if isinstance(p, dict):
            players[p.get("RelGUID")] = {
                "naam": p.get("Naam"),
                "rugnr": p.get("RugNr"),
                "team_kant": "Uit",
                "functie": p.get("Functie"),
            }
        
    # Doorloop alle gebeurtenissen en filter op fouten (GebType == 30)
    foul_records = []
    geb_list = data_vgng.get("GebNis") or []
    for g in geb_list:
        if isinstance(g, dict) and g.get("GebType") == 30:  # 30 = Fout
            rel = g.get("RelGUID")
            p_info = players.get(rel, {
                "naam": f"Onbekend / Bank ({rel})" if rel else "Bank / Onbekend",
                "rugnr": g.get("RugNr", ""),
                "team_kant": "Thuis" if g.get("TofU") == "T" else "Uit",
                "functie": "S",
            })
            
            raw_code = str(g.get("Text", "")).strip()
            clean_code = raw_code.strip("()")
            
            # Bepaal categorie en omschrijving van de fout
            if clean_code.startswith("P"):
                cat = "Persoonlijk (P)"
            elif clean_code.startswith("T"):
                cat = "Technisch (T)"
            elif clean_code.startswith("FL"):
                cat = "Flopping (FL)"
            elif clean_code.startswith("C"):
                cat = "Technisch Coach (C)"
            elif clean_code.startswith("B"):
                cat = "Technisch Bank (B)"
            elif clean_code.startswith("U"):
                cat = "Onsportief (U)"
            elif clean_code.startswith("D") or clean_code.startswith("GD"):
                cat = "Diskwalificerend (D)"
            else:
                cat = "Overig"
                
            omschr = FOUL_EXPLANATIONS.get(clean_code, FOUL_EXPLANATIONS.get(raw_code, f"Fout ({raw_code})"))
            
            foul_records.append({
                "wedguid": wedguid,
                "RelGUID": rel,
                "Naam": p_info["naam"],
                "RugNr": p_info["rugnr"],
                "TeamKant": p_info["team_kant"],
                "FoutType": clean_code,
                "FoutTypeRaw": raw_code,
                "FoutCategorie": cat,
                "Omschrijving": omschr,
                "Periode": g.get("Periode"),
                "Minuut": g.get("Minuut"),
            })
            
    return foul_records

print("Functie parse_match_fouls gereed.")

### Test op één wedstrijd & Referentietabel van alle Fouttypes

Hieronder tonen we eerst de **referentietabel met alle officiële fouttypes van Basketbal Vlaanderen**.
Vervolgens testen we op een wedstrijd-GUID (bv. `BVBL26279100LAHSE21ACA`, die zowel Technisch, Bank, Diskwalificerend als Persoonlijk bevat).

In [ ]:
# 1. Referentietabel van alle mogelijke foutcodes in Basketbal Vlaanderen (VBL / FIBA)
VBL_FOUL_TYPES = [
    {"Code": "P", "Categorie": "Persoonlijk (P)", "Omschrijving": "Persoonlijke fout zonder vrijworpen (balinworp)"},
    {"Code": "P1", "Categorie": "Persoonlijk (P)", "Omschrijving": "Persoonlijke fout met 1 vrijworp (and-one raak shot)"},
    {"Code": "P2", "Categorie": "Persoonlijk (P)", "Omschrijving": "Persoonlijke fout met 2 vrijworpen (op 2-punter of teamfouten)"},
    {"Code": "P3", "Categorie": "Persoonlijk (P)", "Omschrijving": "Persoonlijke fout met 3 vrijworpen (fout op driepunter)"},
    {"Code": "T / T1", "Categorie": "Technisch (T)", "Omschrijving": "Technische fout op speler (1 vrijworp + balbezit)"},
    {"Code": "FL / FL2", "Categorie": "Flopping (FL)", "Omschrijving": "Flopping technische fout (theatrale val, 1 vrijworp)"},
    {"Code": "C / C1", "Categorie": "Technisch Coach (C)", "Omschrijving": "Technische fout direct op de coach voor eigen gedrag"},
    {"Code": "B / B1", "Categorie": "Technisch Bank (B)", "Omschrijving": "Technische fout op de bank / assistent / reservespelers"},
    {"Code": "U / U1 / U2", "Categorie": "Onsportief (U)", "Omschrijving": "Onsportieve fout (harde fout; 2 vrijworpen + balbezit)"},
    {"Code": "D / DI / DI2", "Categorie": "Diskwalificerend (D)", "Omschrijving": "Diskwalificerende fout (onmiddellijke uitsluiting en zaal verlaten)"},
    {"Code": "GD", "Categorie": "Uitsluiting (GD)", "Omschrijving": "Game Disqualification (uitsluiting na 5 fouten of 2x T/U)"},
]

print("=== REFERENTIETABEL: ALLE MOGELIJKE VBL FOUTTYPES ===")
display(pd.DataFrame(VBL_FOUL_TYPES))

# 2. Testen op een wedstrijd-GUID
# Voorbeelden met diverse fouttypes uit het seizoen:
# - "BVBL26279100LAHSE21ACA" -> Waregem vs Haantjes: bevat T1 (Technisch), B1 (Bank), DI2 (Diskwalificerend), P, P1, P2
# - "BVBL26279100LAHSE21AAJ" -> Haantjes vs Izegem: bevat FL2 (Flopping), T1, P, P1, P2
# - "BVBL26279100BLAHSEPEBD" -> Haantjes vs Knokke: bevat C1 (Coach), P, P1, P2
# - "BVBL26279100LAHSE21AAN" -> Haantjes vs Avanti: bevat DI (Diskwalificerend), P, P2
TEST_WEDGUID = "BVBL26279100LAHSE21ACA"  # Pas gerust aan naar een andere wedguid

test_fouls = parse_match_fouls(TEST_WEDGUID)
df_test = pd.DataFrame(test_fouls)

print(f"\n=== Resultaten voor testwedstrijd {TEST_WEDGUID} ===")
if df_test.empty:
    print(f"⚠️ Geen digitale foutenregistratie (DWF) gevonden voor wedstrijd {TEST_WEDGUID}.")
    print("Mogelijke redenen:")
    print("- Voor deze wedstrijd werd geen digitaal wedstrijdblad (DWF play-by-play) bijgehouden.")
    print("- De wedstrijd is nog niet gespeeld of werd forfait gegeven.")
    print("Kies een andere wedstrijd-GUID uit 'df_played', bv: BVBL26279100LAHSE21ACA")
else:
    print(f"Totaal aantal geregistreerde fouten: {len(df_test)}")
    display(df_test[["TeamKant", "Naam", "RugNr", "FoutType", "FoutCategorie", "Omschrijving", "Periode", "Minuut"]].head(10))
    
    print("\n📊 Aantal fouten per speler per specifiek FOUTTYPE (P, P1, P2, T1, B1, DI2, ...):")
    display(pd.crosstab(
        index=[df_test["TeamKant"], df_test["Naam"], df_test["RugNr"]],
        columns=df_test["FoutType"],
        margins=True,
        margins_name="Totaal"
    ))
    
    print("\n📊 Aantal fouten per speler per FOUTCATEGORIE (Persoonlijk, Technisch, Bank, Diskwalificerend, ...):")
    display(pd.crosstab(
        index=[df_test["TeamKant"], df_test["Naam"], df_test["RugNr"]],
        columns=df_test["FoutCategorie"],
        margins=True,
        margins_name="Totaal"
    ))

## Stap 6: Alle voorbije wedstrijden doorlopen en fouten per speler aggregeren

Nu doorlopen we **alle gespeelde wedstrijden** van de geselecteerde ploeg. We verzamelen alle fouten en bouwen een overzichtstabel per speler van onze club.

In [ ]:
all_season_fouls = []

print(f"Bezig met verwerken van {len(df_played)} wedstrijden voor {selected_team_name}...")
for idx, row in df_played.iterrows():
    w_guid = row["wedguid"]
    w_datum = row["datum"]
    w_thuis = row["thuisploeg"]
    w_uit = row["uitploeg"]
    
    # Bepaal of onze ploeg thuis of uit speelde
    is_own_home = (SELECTED_TEAM_GUID in row.get("tTGUID", "")) or ("Haantjes" in w_thuis)
    own_side = "Thuis" if is_own_home else "Uit"
    
    fouls = parse_match_fouls(w_guid)
    for f in fouls:
        f["datum"] = w_datum
        f["match_label"] = f"{w_thuis} vs {w_uit}"
        f["is_eigen_speler"] = (f["TeamKant"] == own_side)
        all_season_fouls.append(f)

df_all_fouls = pd.DataFrame(all_season_fouls)
print(f"Klaar! Totaal geregistreerde fouten: {len(df_all_fouls)}")

### Volledig Overzicht: Fouten per speler van de eigen ploeg

We filteren op de spelers van de eigen club en maken twee overzichtelijke tabellen:
1. **Per specifiek FoutType** (`P`, `P1`, `P2`, `P3`, `T1`, `FL2`, `C1`, `B1`, `DI`, `DI2`, ...)
2. **Samengevat per Hoofdcategorie** (`Persoonlijk`, `Technisch`, `Flopping`, `Coach/Bank`, `Diskwalificerend`, `TOTAAL`)

In [ ]:
# Filter enkel op spelers van onze eigen ploeg
df_own_fouls = df_all_fouls[df_all_fouls["is_eigen_speler"]].copy()

if df_own_fouls.empty:
    print("Geen fouten gevonden voor de eigen ploeg.")
else:
    # 1. Pivot tabel per speler x individueel fouttype
    pivot_player_fouls = pd.crosstab(
        index=df_own_fouls["Naam"],
        columns=df_own_fouls["FoutType"],
        margins=False
    )
    
    # Bereken totalen per hoofdcategorie
    p_cols = [c for c in pivot_player_fouls.columns if c.startswith("P")]
    t_cols = [c for c in pivot_player_fouls.columns if c.startswith("T") or c.startswith("C") or c.startswith("B")]
    fl_cols = [c for c in pivot_player_fouls.columns if c.startswith("FL")]
    u_cols = [c for c in pivot_player_fouls.columns if c.startswith("U")]
    d_cols = [c for c in pivot_player_fouls.columns if c.startswith("D")]
    
    summary = pivot_player_fouls.copy()
    summary["Totaal_P (Persoonlijk)"] = summary[p_cols].sum(axis=1) if p_cols else 0
    summary["Totaal_T (Technisch)"] = summary[t_cols].sum(axis=1) if t_cols else 0
    summary["Totaal_FL (Flopping)"] = summary[fl_cols].sum(axis=1) if fl_cols else 0
    summary["Totaal_U (Onsportief)"] = summary[u_cols].sum(axis=1) if u_cols else 0
    summary["Totaal_D (Diskwalificerend)"] = summary[d_cols].sum(axis=1) if d_cols else 0
    summary["TOTAAL"] = (
        summary["Totaal_P (Persoonlijk)"] +
        summary["Totaal_T (Technisch)"] +
        summary["Totaal_FL (Flopping)"] +
        summary["Totaal_U (Onsportief)"] +
        summary["Totaal_D (Diskwalificerend)"]
    )
    
    # Sorteer op Technisch + Flopping + Diskwalificerend eerst (de opvallendste fouten), dan Totaal
    sort_key = summary["Totaal_T (Technisch)"] + summary["Totaal_FL (Flopping)"] + summary["Totaal_D (Diskwalificerend)"]
    summary = summary.assign(_sort=sort_key).sort_values(by=["_sort", "TOTAAL"], ascending=False).drop(columns=["_sort"])
    
    print(f"📊 1. Volledig overzicht per specifiek FoutType voor {selected_team_name}:")
    display(summary)
    
    print(f"\n📊 2. Overzicht samengevat per FoutCategorie voor {selected_team_name}:")
    pivot_cat = pd.crosstab(
        index=df_own_fouls["Naam"],
        columns=df_own_fouls["FoutCategorie"],
        margins=True,
        margins_name="TOTAAL"
    )
    display(pivot_cat)

### Detail van Speciale Fouten (Technisch, Flopping, Diskwalificerend, Onsportief)
Hier lichten we specifiek alle niet-persoonlijke fouten uit, inclusief de datum, wedstrijd, periode en minuut waarop ze zijn opgelopen.

In [ ]:
speciale_fouten = df_own_fouls[df_own_fouls["FoutCategorie"] != "Persoonlijk (P)"].sort_values(by=["FoutCategorie", "Naam", "datum"])

if speciale_fouten.empty:
    print("Geen technische, flopping, onsportieve of diskwalificerende fouten geregistreerd voor deze ploeg! Fair play!")
else:
    print(f"Aantal speciale fouten (Technisch / Flopping / Diskwalificerend / Onsportief): {len(speciale_fouten)}")
    display(speciale_fouten[["datum", "Naam", "RugNr", "FoutType", "FoutCategorie", "Omschrijving", "Periode", "Minuut", "match_label"]])

## Stap 7: Visualisatie

We visualiseren de verdeling van fouten over de spelers en wedstrijden.

In [ ]:
if not df_own_fouls.empty and not summary.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    top_players = summary.head(12)
    
    # Gestapelde staafdiagram per categorie
    bars_p = ax.bar(top_players.index, top_players["Totaal_P (Persoonlijk)"], label="Persoonlijk (P)", color="#1f77b4")
    bottom_u = top_players["Totaal_P (Persoonlijk)"]
    bars_u = ax.bar(top_players.index, top_players["Totaal_U (Onsportief)"], bottom=bottom_u, label="Onsportief (U)", color="#ff7f0e")
    bottom_t = bottom_u + top_players["Totaal_U (Onsportief)"]
    bars_t = ax.bar(top_players.index, top_players["Totaal_T (Technisch)"] + top_players["Totaal_FL (Flopping)"], bottom=bottom_t, label="Technisch & Flopping", color="#d62728")
    
    ax.set_title(f"Foutenverdeling per speler — {selected_team_name}", fontsize=14, fontweight="bold", pad=15)
    ax.set_ylabel("Aantal fouten", fontsize=12)
    ax.set_xlabel("Speler", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Stap 8: Detailverslag van één specifieke wedstrijd (zoals op Matchdetailpdf)

Hieronder kan je voor eender welke wedstrijd-GUID het volledige wedstrijdblad oproepen en de foutenlijst van beide ploegen inspecteren.

In [ ]:
def toon_uitgebreid_verslag(wedguid: str):
    """Toont het wedstrijdblad en alle fouten van beide ploegen voor een specifieke wedstrijd."""
    fouls = parse_match_fouls(wedguid)
    if not fouls:
        print(f"⚠️ Geen digitaal verslag of fouten gevonden voor {wedguid}")
        return
        
    df = pd.DataFrame(fouls)
    print(f"=== Wedstrijdverslag voor {wedguid} ({len(df)} fouten) ===")
    
    for team_kant in ["Thuis", "Uit"]:
        team_f = df[df["TeamKant"] == team_kant]
        print(f"\n--- {team_kant}ploeg ({len(team_f)} fouten) ---")
        if team_f.empty:
            print("Geen fouten geregistreerd.")
        else:
            pivot = pd.crosstab(
                index=[team_f["Naam"], team_f["RugNr"]],
                columns=team_f["FoutType"],
                margins=True,
                margins_name="Totaal"
            )
            display(pivot)

# Voorbeeld met wedstrijd met technische & bankfouten:
toon_uitgebreid_verslag("BVBL26279100LAHSE21ACA")

## Samenvatting

### Q&A
- **Waarom toont een bepaalde wedstrijd soms 0 fouten?**
  In sommige wedstrijden (bv. lagere jeugd of wanneer er een papieren wedstrijdblad werd gebruikt in plaats van de live digitale DWF) zijn er geen `GebNis`-events opgeslagen in de VBL WebService.
- **Welke fouttypes worden herkend?**
  Alle officiële FIBA/VBL fouttypes: `P`, `P1`, `P2`, `P3`, `T1`, `FL2` (flopping), `C1` (coach), `B1` (bank), `U1`, `U2` (onsportief), `DI`, `DI2`, `D1`, `D2` (diskwalificerend) en `GD` (uitsluiting).

### Data Analysis Key Findings
- De API levert real-time toegang tot het officiële wedstrijdblad zonder dat authenticatie vereist is (`authorization: 'na'`, `ssoBearer: 'ssoWQAndre'`).
- Zowel persoonlijke als technische, flopping, coach, bank en diskwalificerende fouten worden geregistreerd per speler, inclusief periode en minuut.